# Serving an Open LLM with vLLM on Colab

This notebook shows how to serve an open-weight model as an OpenAI-compatible API from a Colab GPU, and how to expose it safely to multiple users.

**Architecture**:
```
students ──ngrok──▶ FastAPI proxy (:8080) ──▶ vLLM (:8000, localhost only)
                    │ per-student API key
                    │ per-key rate limiting
                    └ global concurrency cap
```

Three design decisions worth understanding:

- **vLLM is never exposed directly.** The inference server has no authentication of its own; the proxy is the only entry point, and vLLM listens on localhost only.
- **One API key per student.** A runaway loop in one notebook exhausts that student's rate budget, not everyone else's — and the proxy logs show who is generating traffic.
- **A global concurrency cap protects the GPU.** Rate limiting stops slow loops; the concurrency cap is what prevents the KV cache from being overwhelmed by queued requests.

Model: `Qwen3-Coder-30B-A3B-Instruct-FP8` (~31 GB on a 40 GB A100, Marlin kernels on Ampere). If you need more context / KV cache headroom, switch to the 4-bit variant in the configuration form.


## 1. Dependencies

In [ ]:
%%capture
# Only vLLM and the proxy are named here, on purpose. vLLM pins its own torch,
# transformers, tokenizers and huggingface_hub, and naming those independently
# is what broke this cell before: pip honoured vLLM's huggingface_hub and our
# transformers at the same time, and the two do not fit. The result was an
# ImportError raised by `vllm serve`, after the weights had already downloaded.
#
# So the versions below are deliberately incomplete. The next cell prints what
# the resolver actually chose; those numbers belong here, pinned, before this
# notebook is handed to anyone.
!pip install -qq \
    vllm==0.18.0 \
    fastapi==0.135.2 \
    uvicorn==0.42.0 \
    pyngrok==7.5.1 \
    httpx


In [ ]:
# The install above is captured, and pip signals a bad resolution on stdout
# rather than through its exit code, so a broken stack leaves no trace until
# vLLM starts -- minutes and 31 GB of weights later. Prove it imports here
# instead, in a fresh interpreter: the same thing `vllm serve` will be, and
# unaffected by whatever this kernel already imported.
import subprocess
import sys

_probe = (
    "import torch, transformers, tokenizers, huggingface_hub, vllm\n"
    "for _m in (torch, transformers, tokenizers, huggingface_hub, vllm):\n"
    "    print(f'{_m.__name__}=={_m.__version__}')\n"
)
_check = subprocess.run([sys.executable, "-c", _probe], capture_output=True, text=True)
print(_check.stdout, end="")
if _check.returncode != 0:
    print(_check.stderr, end="")
    raise RuntimeError("the installed stack does not import -- see the traceback above")

# pip reports version conflicts it resolved against, which the import above
# cannot see: a package can import fine and still sit next to a dependency
# whose declared range it violates.
!pip check

# TODO: copy the five versions printed above into the install cell, pin them,
# and delete this note. Until then this notebook resolves differently on
# different days, which is exactly what the course cannot afford.


## 2. Imports

In [2]:
from __future__ import annotations

import os
import signal
import socket
import subprocess
import time
from contextlib import contextmanager
from dataclasses import dataclass
from threading import Thread
from typing import Generator, List, Optional, Sequence

import torch
from pyngrok import ngrok
from openai import OpenAI
from huggingface_hub import login, snapshot_download


## 3. Secrets

Two credentials: `HF_TOKEN` and `NGROK_TOKEN`. Paste them into the cell below.

The ngrok *domain* is not a credential — it is the address students type into
their own client — so it sits in the configuration section instead, where it can
be read.

On the Colab web page the Secrets panel is the better option, and the cell has
the two lines for it commented out. That panel belongs to the web page, not to
the runtime: driving the same runtime from an IDE plugin leaves `userdata`
undefined, which is why the pasted values are the default here.


In [3]:
# Two Colab Secrets, set once from the key icon in the left sidebar and enabled
# for this notebook. They are read rather than pasted on purpose: a token typed
# into a cell is saved with the notebook, and this one is in a repository.
#
#   HF_TOKEN     https://huggingface.co/settings/tokens
#                fine-grained, read-only, limited to the model repo below
#   NGROK_TOKEN  https://dashboard.ngrok.com -> Authtokens
#
# Driving the runtime from an IDE plugin instead of the Colab web page gives no
# `userdata`; there, export the two as environment variables before launching.
from google.colab import userdata
HF_TOKEN, NGROK_TOKEN = userdata.get("HF_TOKEN"), userdata.get("NGROK_TOKEN")

assert HF_TOKEN and NGROK_TOKEN, (
    "set HF_TOKEN and NGROK_TOKEN as Colab Secrets (key icon, left sidebar) "
    "and switch both on for this notebook")

os.environ["HF_TOKEN"] = HF_TOKEN
login(HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)
print("hugging face and ngrok authenticated")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


hugging face and ngrok authenticated                                                                


## 4. Configuration

In [4]:
MODEL = 'Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8' # @param {type:'string'}
# Pinned to a commit, not to a branch name: a Hugging Face repo can be updated
# under the same name, and a course that runs for nine weeks would then serve a
# different model to different people without anyone noticing.
REVISION = 'dcaee4d4dfc5ee71ad501f01f530e5652438fde0' # @param {type:'string'}

# 4-bit alternative for 24 GB cards (L4): fits where FP8 does not, lower quality.
# MODEL = 'Qwen/Qwen3-Coder-30B-A3B-Instruct-AWQ'
# REVISION = None  # look up the commit for this repo before using it

# Tool calling is off in vLLM unless it is asked for, and an OpenAI-style
# request with tools then fails at 400 rather than degrading to plain text. The
# parser is model-specific: it is what turns the model's own tool-call syntax
# into the OpenAI schema. 'hermes' is the fallback if this one is rejected.
TOOL_CALL_PARSER = 'qwen3_coder' # @param {type:'string'}

# One reserved domain per service, not per session: students paste this address
# into their own client, so it has to survive every restart of this runtime for
# the whole course. A second service gets its own name (kgtp-neo4j, ...) rather
# than reusing this one. Not a secret — overridable from the environment for a
# one-off test.
NGROK_DOMAIN = os.environ.get("NGROK_DOMAIN") or 'kgtp-llm.ngrok-free.app' # @param {type:'string'}

VLLM_PORT = 8000      # @param {type:'integer'}
# Deliberately not 8080. Both ports are internal — ngrok publishes on 443 — so
# there is nothing to gain from a common number and something to lose: on a
# hosted runtime the popular ports are the ones already spoken for.
PROXY_PORT = 8899     # @param {type:'integer'}
MAX_MODEL_LEN = 32768 # @param {type:'integer'}
GPU_MEM_UTIL = 0.92   # @param {type:'number'}

# API keys: one per student. The name shows up in the proxy logs.
API_KEYS = {
    "student-key-01": "alice",
    "student-key-02": "bob",
    "teacher-key": "giuseppe",
}
RATE_LIMIT_RPM = 10   # requests/minute per key
MAX_CONCURRENT = 4    # in-flight requests toward the GPU

print(f"model:  {MODEL}@{REVISION[:8] if REVISION else 'unpinned'}")
print(f"domain: {NGROK_DOMAIN}")
print(f"ports:  vLLM {VLLM_PORT}, proxy {PROXY_PORT}")
print(f"tools:  {TOOL_CALL_PARSER}")


model:  Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8@dcaee4d4
domain: kgtp-llm.ngrok-free.app
ports:  vLLM 8000, proxy 8899
tools:  qwen3_coder


## 5. Download the weights

Done here, on its own, rather than left to vLLM at startup.

vLLM opens its port only once the model is loaded, so a first launch has to
fetch ~31 GB before anything answers. Whoever is waiting on that port cannot
tell a slow download from a hung server, and a timeout kills the download
halfway. Pulling the weights in a separate step makes the progress visible, and
`snapshot_download` resumes rather than restarting.

It is also the cell that decides whether the rest of the notebook can work at
all: the wrong GPU fails here, cheaply, instead of after the launch arguments
have been assembled.


In [5]:
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Change the Colab runtime type to a GPU.")

gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}, {total_gb:.0f} GB")

# ~31 GB of weights plus the KV cache: below roughly 38 GB the server starts and
# then dies on an out-of-memory error minutes later, which is a slow way to find
# out. Say it now instead.
if total_gb < 38:
    print(
        f"\nWARNING: {total_gb:.0f} GB is not enough for the FP8 weights.\n"
        "Use the AWQ 4-bit variant in the configuration cell, or switch to an A100."
    )

local_path = snapshot_download(
    repo_id=MODEL,
    revision=REVISION,
    token=HF_TOKEN,
    max_workers=8,
)
print("weights at", local_path)


GPU: NVIDIA A100-SXM4-80GB, 79 GB


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

qwen3coder_tool_parser.py: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

weights at /root/.cache/huggingface/hub/models--Qwen--Qwen3-Coder-30B-A3B-Instruct-FP8/snapshots/dcaee4d4dfc5ee71ad501f01f530e5652438fde0


## 6. vLLM helpers

Small utilities to start vLLM as a subprocess, stream its logs into the cell output, and shut it down cleanly. vLLM listens on `127.0.0.1` only.

In [6]:
@dataclass
class VLLMConfig:
    model: str = MODEL
    revision: Optional[str] = REVISION
    host: str = "127.0.0.1"   # never exposed: the proxy sits in front
    port: int = VLLM_PORT
    max_model_len: int = MAX_MODEL_LEN
    gpu_memory_utilization: float = GPU_MEM_UTIL
    trust_remote_code: bool = True
    tool_call_parser: Optional[str] = TOOL_CALL_PARSER

    @property
    def base_url(self) -> str:
        return f"http://{self.host}:{self.port}"


def select_dtype() -> str:
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        return "bfloat16" if major >= 8 else "float16"
    return "float32"


def free_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    free, _total = torch.cuda.mem_get_info()
    return free / 1024**3


def wait_for_port(
    host: str,
    port: int,
    timeout: float = 1800.0,
    interval: float = 0.5,
    proc: Optional["VLLMProcess"] = None,
) -> bool:
    """Wait until something accepts connections on host:port.

    Give up early if the process we are waiting for has already exited: without
    this check a crash on startup is indistinguishable from a slow load, and the
    caller blocks for the full timeout before reporting a useless error.
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc is not None and proc.proc.poll() is not None:
            print(f"process exited with code {proc.proc.returncode} before opening the port")
            return False
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1.0)
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(interval)
    return False


def build_args(cfg: VLLMConfig, dtype: str) -> Sequence[str]:
    args = [
        "vllm", "serve", cfg.model,
        "--dtype", dtype,
        "--max-model-len", str(cfg.max_model_len),
        "--gpu-memory-utilization", str(cfg.gpu_memory_utilization),
        "--host", cfg.host,
        "--port", str(cfg.port),
    ]
    if cfg.revision:
        args += ["--revision", cfg.revision]
    if cfg.tool_call_parser:
        # Without both flags an OpenAI request carrying "tools" is
        # rejected with a 400 -- it does not fall back to plain text.
        # The agentic scenarios of module 5 need this; the text2cypher
        # ones never touch it, which is why it is easy to miss.
        args += ["--enable-auto-tool-choice",
                 "--tool-call-parser", cfg.tool_call_parser]
    if cfg.trust_remote_code:
        args.append("--trust-remote-code")
    return args


def _pump(pipe, tag: str, sink: Optional[List[str]] = None):
    try:
        for line in iter(pipe.readline, ""):
            if not line:
                break
            print(f"[vLLM {tag}] {line}", end="")
            if sink is not None:
                sink.append(f"[{tag}] {line}")
    except Exception as e:
        print(f"[vLLM LOG ERROR {tag}] {e}")
    finally:
        try:
            pipe.close()
        except Exception:
            pass


class VLLMProcess:
    def __init__(self, proc: subprocess.Popen):
        self.proc = proc
        self.logs: List[str] = []
        self._t_out = Thread(target=_pump, args=(proc.stdout, "OUT", self.logs), daemon=True)
        self._t_err = Thread(target=_pump, args=(proc.stderr, "ERR", self.logs), daemon=True)
        self._t_out.start()
        self._t_err.start()

    @property
    def pid(self) -> int:
        return self.proc.pid


def start_vllm(cfg: VLLMConfig) -> VLLMProcess:
    proc = subprocess.Popen(
        build_args(cfg, select_dtype()),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        start_new_session=True,
        bufsize=1,
        env=os.environ.copy(),
    )
    return VLLMProcess(proc)


def stop_vllm(p: Optional[VLLMProcess], verbose: bool = True) -> None:
    """Stop vLLM and everything it forked.

    vLLM runs its engine in child processes, and those children are the ones
    holding the weights. Signalling only the parent leaves them alive and the
    VRAM allocated, so the next launch fails on an out-of-memory error that
    looks like a configuration problem and is not. `start_new_session=True` in
    start_vllm put the whole tree in one process group, which is what makes it
    addressable here.
    """
    if not p:
        return
    try:
        pgid = os.getpgid(p.proc.pid)
    except (ProcessLookupError, PermissionError):
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        try:
            p.proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(pgid, signal.SIGKILL)
            p.proc.wait(timeout=10)
    except Exception as e:
        print(f"error while stopping vLLM: {e}")
    if verbose and torch.cuda.is_available():
        time.sleep(2)  # the driver reclaims memory slightly after the processes die
        print(f"vLLM stopped, {free_vram_gb():.1f} GB of VRAM free")


@contextmanager
def run_vllm(cfg: VLLMConfig) -> Generator[tuple, None, None]:
    proc = start_vllm(cfg)
    try:
        if not wait_for_port(cfg.host, cfg.port, proc=proc):
            stop_vllm(proc)
            raise RuntimeError("vLLM didn't become ready on time.")
        print(f"vLLM up at {cfg.base_url} (pid={proc.pid})")
        yield (cfg.base_url, proc)
    finally:
        stop_vllm(proc)


## 7. FastAPI proxy

Written to a file and launched as a subprocess (same pattern as vLLM). It reads its configuration from `proxy_config.json`.

In [7]:
import json as _json
with open("/content/proxy_config.json", "w") as f:
    _json.dump({
        "vllm_url": f"http://127.0.0.1:{VLLM_PORT}",
        "api_keys": API_KEYS,
        "rate_limit_rpm": RATE_LIMIT_RPM,
        "max_concurrent": MAX_CONCURRENT,
    }, f)
print("proxy_config.json written")


proxy_config.json written


In [8]:
%%writefile /content/vllm_proxy.py
"""Educational reverse proxy for vLLM: per-key auth, rate limiting, concurrency cap."""
import asyncio
import json
import time
from collections import defaultdict, deque

import httpx
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse, JSONResponse

CFG = json.load(open("/content/proxy_config.json"))
API_KEYS = CFG["api_keys"]
RATE_LIMIT_RPM = CFG["rate_limit_rpm"]
MAX_CONCURRENT = CFG["max_concurrent"]

app = FastAPI(title="Educational vLLM proxy")
semaphore = asyncio.Semaphore(MAX_CONCURRENT)
request_log = defaultdict(deque)
client = httpx.AsyncClient(base_url=CFG["vllm_url"], timeout=300)


def check_auth(request: Request) -> str:
    auth = request.headers.get("authorization", "")
    if not auth.startswith("Bearer "):
        raise HTTPException(401, "Missing Authorization: Bearer <key> header")
    key = auth.removeprefix("Bearer ").strip()
    if key not in API_KEYS:
        raise HTTPException(403, "Invalid API key")
    return key


def check_rate_limit(key: str) -> None:
    now = time.monotonic()
    window = request_log[key]
    while window and now - window[0] > 60:
        window.popleft()
    if len(window) >= RATE_LIMIT_RPM:
        retry_in = int(60 - (now - window[0])) + 1
        raise HTTPException(429, f"Limit of {RATE_LIMIT_RPM} req/min exceeded. Retry in {retry_in}s.")
    window.append(now)


@app.get("/health")
async def health():
    return {"status": "ok", "in_flight": MAX_CONCURRENT - semaphore._value}


@app.post("/v1/chat/completions")
@app.post("/v1/completions")
async def proxy(request: Request):
    key = check_auth(request)
    check_rate_limit(key)
    student = API_KEYS[key]

    body = await request.body()
    # Parse the body instead of searching it for a substring: a client that
    # serialises without a space after the colon, or that reorders the keys,
    # would defeat a substring test and get its stream handled as a plain
    # response.
    try:
        is_stream = bool(json.loads(body).get("stream", False))
    except Exception:
        raise HTTPException(400, "Body is not valid JSON")

    try:
        await asyncio.wait_for(semaphore.acquire(), timeout=30)
    except asyncio.TimeoutError:
        raise HTTPException(503, "Server is saturated, retry in a few seconds.")

    print(f"[{time.strftime('%H:%M:%S')}] {student} -> {request.url.path} (stream={is_stream})", flush=True)

    # asyncio.Semaphore has no ceiling: releasing more times than we acquired
    # raises the concurrency cap permanently instead of erroring, so a handful
    # of failed requests would quietly remove the protection the cap exists for.
    released = False

    def release_once() -> None:
        nonlocal released
        if not released:
            released = True
            semaphore.release()

    # Set once the response body becomes the streaming generator's
    # responsibility: from that point the release happens when the stream ends,
    # not when this function returns.
    stream_owns_release = False
    try:
        if is_stream:
            upstream_request = client.build_request(
                "POST", request.url.path, content=body,
                headers={"content-type": "application/json"},
            )
            upstream = await client.send(upstream_request, stream=True)

            async def stream_and_release():
                try:
                    async for chunk in upstream.aiter_bytes():
                        yield chunk
                finally:
                    await upstream.aclose()
                    release_once()

            stream_owns_release = True
            return StreamingResponse(
                stream_and_release(),
                status_code=upstream.status_code,
                media_type="text/event-stream",
            )
        else:
            resp = await client.post(
                request.url.path, content=body,
                headers={"content-type": "application/json"},
            )
            return JSONResponse(resp.json(), status_code=resp.status_code)
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(502, f"Error reaching vLLM: {e}")
    finally:
        if not stream_owns_release:
            release_once()


@app.get("/v1/models")
async def models(request: Request):
    check_auth(request)
    resp = await client.get("/v1/models")
    return JSONResponse(resp.json(), status_code=resp.status_code)


Writing /content/vllm_proxy.py


In [9]:
def port_is_taken(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1.0)
        return s.connect_ex((host, port)) == 0


def pids_on_port(port: int) -> List[int]:
    out = subprocess.run(["lsof", "-ti", f":{port}"], capture_output=True, text=True)
    return [int(p) for p in out.stdout.split() if p.strip().isdigit()]


def cmdline_of(pid: int) -> str:
    try:
        with open(f"/proc/{pid}/cmdline", "rb") as f:
            return f.read().replace(b"\0", b" ").decode(errors="replace").strip()
    except OSError:
        return ""


def free_port(port: int) -> None:
    """Free `port`, but only from a proxy we started ourselves.

    A leftover proxy from an interrupted run is worse than no proxy at all: the
    port answers, every readiness check passes, and the tunnel gets published in
    front of old code. It still has to be cleared by identifying the process,
    never by killing whatever holds the number — on a hosted runtime that port
    may belong to infrastructure we know nothing about, and killing it takes the
    session down with it.
    """
    if not port_is_taken(port):
        return
    for pid in pids_on_port(port):
        cmd = cmdline_of(pid)
        if "vllm_proxy" not in cmd:
            raise RuntimeError(
                f"port {port} is held by pid {pid}, which is not our proxy:\n"
                f"    {cmd or '(unknown command)'}\n"
                f"Change PROXY_PORT in the configuration section rather than killing it."
            )
        print(f"killing leftover proxy, pid {pid}")
        os.kill(pid, signal.SIGKILL)
    time.sleep(1)
    if port_is_taken(port):
        raise RuntimeError(f"port {port} is still busy")


def start_proxy() -> subprocess.Popen:
    free_port(PROXY_PORT)
    proc = subprocess.Popen(
        ["uvicorn", "vllm_proxy:app", "--host", "127.0.0.1", "--port", str(PROXY_PORT)],
        cwd="/content",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )
    Thread(target=_pump, args=(proc.stdout, "PROXY"), daemon=True).start()
    return proc


def stop_proxy(proc: Optional[subprocess.Popen]) -> None:
    """Same process-group teardown as stop_vllm, and for the same reason:
    signalling only the parent leaves uvicorn holding the port."""
    if not proc:
        return
    try:
        pgid = os.getpgid(proc.pid)
    except (ProcessLookupError, PermissionError):
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(pgid, signal.SIGKILL)
            proc.wait(timeout=5)
    except Exception as e:
        print(f"error while stopping the proxy: {e}")


def proxy_is_healthy(port: int, proc: subprocess.Popen, quiet: bool = False) -> bool:
    """An open port is not proof that our proxy is behind it.

    The process we started has to still be alive *and* /health has to answer.
    Either one alone can be true while the endpoint is broken.
    """
    import json as _j
    import urllib.request

    if proc.poll() is not None:
        if not quiet:
            print(f"the proxy exited with code {proc.returncode}")
        return False
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5) as r:
            return _j.loads(r.read()).get("status") == "ok"
    except Exception as e:
        if not quiet:
            print(f"/health did not answer: {e}")
        return False


## 8. Local test (optional, no tunnel)

Starts vLLM, runs one sanity-check call, shuts everything down.

**Skip this on a 40 GB A100.** Loading ~31 GB of weights takes minutes, and this
cell does it once only to throw the server away — section 9 does the same load
and then keeps it. Run it only when changing the launch arguments, where a fast
failure is worth the wait. Set `RUN_LOCAL_TEST = True` below to enable it.


In [10]:
RUN_LOCAL_TEST = False  # @param {type:'boolean'}

if not RUN_LOCAL_TEST:
    print("skipped: section 9 loads the same model and keeps it serving")
else:
    cfg = VLLMConfig()
    with run_vllm(cfg) as (base_url, proc):
        openai_client = OpenAI(api_key="EMPTY", base_url=f"{base_url}/v1")
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Write a Cypher query that counts nodes per label."}],
            temperature=0,
        )
        print(resp.choices[0].message.content)


skipped: section 9 loads the same model and keeps it serving


## 9. Public serving

vLLM + proxy + ngrok tunnel on the **proxy** (never on vLLM). The cell keeps running until you stop it.

In [ ]:
cfg = VLLMConfig()
print(f"{free_vram_gb():.1f} GB of VRAM free before starting")

vllm_proc = start_vllm(cfg)
proxy_proc = None
try:
    if not wait_for_port(cfg.host, cfg.port, proc=vllm_proc):
        # The log is streamed above, but by the time the traceback appears it has
        # scrolled far out of view. Repeat the tail here so the error names its
        # own cause instead of pointing at output nobody scrolls back to.
        tail = "".join(vllm_proc.logs[-40:]) or "(vLLM printed nothing at all)"
        raise RuntimeError("vLLM did not open the port. Last lines of its log:\n" + tail)
    print(f"vLLM up at {cfg.base_url}, {free_vram_gb():.1f} GB of VRAM still free")

    # The tunnel goes up only once the proxy has been proved to be ours and
    # answering. Opening it earlier can publish a stale process left behind by
    # an interrupted run, which from the outside is indistinguishable from
    # success.
    proxy_proc = start_proxy()
    healthy = False
    deadline = time.time() + 30
    while time.time() < deadline:
        if proxy_proc.poll() is not None:
            raise RuntimeError("the proxy died on startup — see the log above")
        if proxy_is_healthy(PROXY_PORT, proxy_proc, quiet=True):
            healthy = True
            break
        time.sleep(1)
    if not healthy:
        proxy_is_healthy(PROXY_PORT, proxy_proc)  # once more, this time reporting why
        raise RuntimeError("the proxy did not become healthy within 30s")
    print(f"proxy healthy on :{PROXY_PORT}")

    public_url = ngrok.connect(PROXY_PORT, "http", domain=NGROK_DOMAIN).public_url
    print("Endpoint for students:", public_url.rstrip("/") + "/v1")
    print("Health check:", public_url.rstrip("/") + "/health")

    while True:
        time.sleep(1)
finally:
    try:
        ngrok.kill()
    except Exception:
        pass
    stop_proxy(proxy_proc)
    stop_vllm(vllm_proc)


78.8 GB of VRAM free before starting
[vLLM ERR] 2026-08-18 13:32:31.548987: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[vLLM ERR] 2026-08-18 13:32:31.619746: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
[vLLM ERR] To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[vLLM OUT] (APIServer pid=2276) INFO 08-18 13:32:48 [utils.py:297] 
[vLLM OUT] (APIServer pid=2276) INFO 08-18 13:32:48 [utils.py:297]        █     █     █▄   ▄█
[vLLM OUT] (APIServer pid=2276) INFO 08-18 13:32:48 [utils.py:297]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.18.0
[vLLM OUT] (APIServer pid=2276) 

## 10. Example client (hand this to the students)

The base URL is the reserved domain, which stays the same across restarts of the
serving runtime — so this snippet does not have to be reissued every session.

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://kgtp-llm.ngrok-free.app/v1",
    api_key="student-key-01",
    default_headers={"ngrok-skip-browser-warning": "1"},
)
resp = client.chat.completions.create(
    model="Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8",
    messages=[{"role": "user", "content": "MATCH ..."}],
)
print(resp.choices[0].message.content)
```

`/health` on the same domain answers without a key, and reports how many
requests are in flight — enough to tell "the endpoint is down" apart from
"someone else is using it", which is the question that actually gets asked
during a lab.
